In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display

BASE_DIR = Path("/Users/ipman/Claude Projects/Maxerience")
PARQUET_GLOB = str(BASE_DIR / "*/*.parquet")
con = duckdb.connect()

def q(sql: str) -> pd.DataFrame:
    full_sql = sql.replace("parquet", f"read_parquet('{PARQUET_GLOB}')")
    return con.execute(full_sql).df()

print("✅ Connected — ready to visualize")

✅ Connected — ready to visualize


---
# 📊 Maxerience Data Explorer
Interactive charts built on your 28 daily parquet files (Feb 2026).

## 1 · Daily Scan Volume

In [2]:
df = q("""
    SELECT
        CAST(SliceStartTime AS DATE) AS scan_date,
        COUNT(*) AS scans
    FROM parquet
    GROUP BY scan_date
    ORDER BY scan_date
""")

fig = px.bar(
    df, x='scan_date', y='scans',
    title='Daily Scan Volume — February 2026',
    labels={'scan_date': 'Date', 'scans': 'Row Count'},
    color='scans', color_continuous_scale='Blues'
)
fig.update_layout(coloraxis_showscale=False, xaxis_tickangle=-45)
fig.show()

## 2 · Top 20 Brands by Shelf Facings

In [3]:
df = q("""
    SELECT
        BrandName,
        COUNT(*) AS facings,
        SUM(SingleFacings) AS total_single_facings
    FROM parquet
    WHERE BrandName IS NOT NULL AND BrandName != ''
    GROUP BY BrandName
    ORDER BY facings DESC
    LIMIT 20
""")

fig = px.bar(
    df.sort_values('facings'), x='facings', y='BrandName',
    orientation='h',
    title='Top 20 Brands by Shelf Facings',
    labels={'facings': 'Facings', 'BrandName': 'Brand'},
    color='facings', color_continuous_scale='Teal'
)
fig.update_layout(coloraxis_showscale=False, height=600)
fig.show()

## 3 · Product Category Mix

In [4]:
df = q("""
    SELECT
        COALESCE(NULLIF(ProductCategory, ''), 'Unknown') AS category,
        COUNT(*) AS cnt
    FROM parquet
    GROUP BY category
    ORDER BY cnt DESC
    LIMIT 15
""")

fig = px.pie(
    df, names='category', values='cnt',
    title='Product Category Mix (Top 15)',
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=550)
fig.show()

## 4 · Empty Shelf Rate by Brand (Top 20)

In [5]:
df = q("""
    SELECT
        BrandName,
        COUNT(*) AS total,
        SUM(CASE WHEN IsEmpty THEN 1 ELSE 0 END) AS empty_count,
        ROUND(100.0 * SUM(CASE WHEN IsEmpty THEN 1 ELSE 0 END) / COUNT(*), 2) AS empty_pct
    FROM parquet
    WHERE BrandName IS NOT NULL AND BrandName != ''
    GROUP BY BrandName
    HAVING total > 10000
    ORDER BY empty_pct DESC
    LIMIT 20
""")

fig = px.bar(
    df.sort_values('empty_pct'), x='empty_pct', y='BrandName',
    orientation='h',
    title='Empty Shelf Rate by Brand (%) — min 10K facings',
    labels={'empty_pct': 'Empty %', 'BrandName': 'Brand'},
    color='empty_pct', color_continuous_scale='Reds'
)
fig.update_layout(coloraxis_showscale=False, height=600)
fig.show()

## 5 · Daily Empty vs Stocked Facings Trend

In [6]:
df = q("""
    SELECT
        CAST(SliceStartTime AS DATE) AS scan_date,
        SUM(CASE WHEN IsEmpty THEN 1 ELSE 0 END) AS empty,
        SUM(CASE WHEN NOT IsEmpty THEN 1 ELSE 0 END) AS stocked
    FROM parquet
    GROUP BY scan_date
    ORDER BY scan_date
""")

df_melted = df.melt(id_vars='scan_date', value_vars=['empty', 'stocked'], var_name='status', value_name='count')

fig = px.area(
    df_melted, x='scan_date', y='count', color='status',
    title='Daily Empty vs Stocked Facings',
    labels={'scan_date': 'Date', 'count': 'Facings', 'status': 'Status'},
    color_discrete_map={'empty': '#ef4444', 'stocked': '#22c55e'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 6 · Shelf Position Heatmap (Shelf × Position)

In [7]:
df = q("""
    SELECT
        Shelf,
        Position,
        COUNT(*) AS facings
    FROM parquet
    WHERE Shelf BETWEEN 1 AND 10
      AND Position BETWEEN 1 AND 30
    GROUP BY Shelf, Position
    ORDER BY Shelf, Position
""")

pivot = df.pivot(index='Shelf', columns='Position', values='facings').fillna(0)

fig = px.imshow(
    pivot,
    title='Shelf Position Heatmap — Facing Density',
    labels=dict(x='Position', y='Shelf', color='Facings'),
    color_continuous_scale='Viridis',
    aspect='auto'
)
fig.update_layout(height=450)
fig.show()

## 7 · Top 15 Products by Facing Count

In [8]:
df = q("""
    SELECT
        ShortName AS product,
        BrandName AS brand,
        COUNT(*) AS facings,
        ROUND(100.0 * SUM(CASE WHEN IsEmpty THEN 1 ELSE 0 END) / COUNT(*), 1) AS empty_pct
    FROM parquet
    WHERE ShortName IS NOT NULL AND ShortName != ''
    GROUP BY ShortName, BrandName
    ORDER BY facings DESC
    LIMIT 15
""")

fig = px.bar(
    df.sort_values('facings'), x='facings', y='product',
    orientation='h', color='brand',
    title='Top 15 Products by Facing Count',
    labels={'facings': 'Total Facings', 'product': 'Product'},
    hover_data=['empty_pct']
)
fig.update_layout(height=600, showlegend=True)
fig.show()
display(df.reset_index(drop=True))

,product,brand,facings,empty_pct
0,Non-beverage - food,Non-beverage,1334482,0.0
1,12Z CN 12FP COKE,Coca-Cola,875472,0.0
2,Empty,NaN,623580,100.0
3,Combo Dr Pepper 12 oz 12pk CAN,Dr Pepper,490611,0.0
4,Non-beverage - Others,Non-beverage,473901,0.0
5,Core Power Chocolate 14 oz PET,Core Power,391295,0.0
6,Core Power Vanilla 14 oz PET,Core Power,359472,0.0
7,Combo Coca-Cola 7.5 oz 10pk CAN,Coca-Cola,295325,0.0
8,12Z CN 12FP SPRITE,Sprite,294168,0.0
9,Core Power BananaStraw 14 oz PET,Core Power,286708,0.0


## 8 · Brand Share Over Time (Top 8 Brands)

In [9]:
# Get top 8 brands first
top_brands = q("""
    SELECT BrandName FROM parquet
    WHERE BrandName IS NOT NULL AND BrandName != ''
    GROUP BY BrandName ORDER BY COUNT(*) DESC LIMIT 8
""")['BrandName'].tolist()

brands_sql = "', '".join(top_brands)

df = q(f"""
    SELECT
        CAST(SliceStartTime AS DATE) AS scan_date,
        BrandName,
        COUNT(*) AS facings
    FROM parquet
    WHERE BrandName IN ('{brands_sql}')
    GROUP BY scan_date, BrandName
    ORDER BY scan_date, BrandName
""")

fig = px.line(
    df, x='scan_date', y='facings', color='BrandName',
    title='Daily Facing Trend — Top 8 Brands',
    labels={'scan_date': 'Date', 'facings': 'Facings', 'BrandName': 'Brand'},
    markers=True
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

## 9 · Fullness Status Breakdown

In [10]:
df = q("""
    SELECT
        COALESCE(NULLIF(FullnessStatus, ''), 'Not Set') AS status,
        COUNT(*) AS cnt
    FROM parquet
    GROUP BY status
    ORDER BY cnt DESC
""")

fig = px.pie(
    df, names='status', values='cnt',
    title='Fullness Status Distribution',
    hole=0.35,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='outside', textinfo='percent+label')
fig.update_layout(height=500)
fig.show()

## 10 · 🔍 Ad-hoc Query
Edit the SQL below to explore the data freely.

In [11]:
# ✏️  Edit this SQL to explore — use 'parquet' as the table name
result = q("""
    SELECT
        ProductCategory,
        BrandName,
        COUNT(*) AS facings,
        ROUND(AVG(CAST(IsEmpty AS INTEGER)) * 100, 1) AS empty_pct
    FROM parquet
    WHERE ProductCategory IS NOT NULL AND ProductCategory != ''
      AND BrandName IS NOT NULL AND BrandName != ''
    GROUP BY ProductCategory, BrandName
    HAVING facings > 5000
    ORDER BY facings DESC
    LIMIT 30
""")

display(result)

,ProductCategory,BrandName,facings,empty_pct
0,Sparkling Soft Drink,Coca-Cola,2914441,0.0
1,Non-Beverage,Non-beverage,1933779,0.0
2,Energy Drinks,Monster,1631150,0.0
3,RTD Nutrition,Core Power,1439674,0.0
4,Sparkling Soft Drink,Dr Pepper,1326422,0.0
5,Sparkling Soft Drink,Sprite,1196553,0.0
6,Energy Drinks,Red Bull,1012364,0.0
7,Sports Drinks,GATORADE,913003,0.0
8,Sparkling Soft Drink,Coca-Cola Zero,874241,0.0
9,Sparkling Soft Drink,PEPSI,840890,0.0
